# 6. Traverse replies

Depth-1 reply traversal over endorsement posts. We re-classify each reply through the same endorsement / flag / irrelevant schema and append the new rows back into `classified.parquet` (and the reply-resolved posts into `resolved_posts.parquet`).

Only **Bluesky and Reddit** endorsements get traversed — other platforms either lack a reply graph or aren't supported by the traverser.

Inputs: `output/<RUN_ID>/{papers,resolved_posts,classified,posts}.parquet`.
Outputs: appended rows in `classified.parquet` + `resolved_posts.parquet`; `stage_6_traverse_replies` in `manifest.json`.

## Setup

Load `.env` (Anthropic + Reddit creds) and pick the latest `RUN_ID` that has the prerequisite parquet files.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

REPO_ROOT = Path.cwd().parent.resolve()
load_dotenv(REPO_ROOT / ".env")

OUTPUT_ROOT = REPO_ROOT / "altendor" / "output"

# Override RUN_ID here if you want a specific historical run instead of the latest.
RUN_ID: str | None = None
if RUN_ID is None:
    candidates = sorted(
        [p.name for p in OUTPUT_ROOT.iterdir() if (p / "classified.parquet").exists()]
    )
    if not candidates:
        raise FileNotFoundError("No run with classified.parquet found. Run notebook 5 first.")
    RUN_ID = candidates[-1]

OUT_DIR = OUTPUT_ROOT / RUN_ID
print(f"Using run id: {RUN_ID}\nOutput dir: {OUT_DIR}")

In [ ]:
import pandas as pd

papers_df = pd.read_parquet(OUT_DIR / "papers.parquet").set_index("ro_id")
resolved_df = pd.read_parquet(OUT_DIR / "resolved_posts.parquet")
classified_df = pd.read_parquet(OUT_DIR / "classified.parquet")
posts_df = pd.read_parquet(OUT_DIR / "posts.parquet")
post_to_ro = dict(zip(posts_df["post_id"], posts_df["ro_id"]))
resolved_lookup = {r["post_id"]: r for _, r in resolved_df.iterrows()}
print(f"{len(classified_df)} classified rows; {len(resolved_lookup)} resolved posts")

## Build traversal seeds

Pick endorsement rows whose resolved post is on Bluesky or Reddit and whose paper context is available.

In [ ]:
from altendor.traverse.replies import TraversalSeed
from altendor.classify.classifier import PaperCtx
from altendor.classify.schema import Endorsement
from altendor.enrich.text_resolver import ResolvedPost

seeds: list[TraversalSeed] = []
endorsements = classified_df[classified_df["kind"] == "endorsement"]
for _, row in endorsements.iterrows():
    resolved = resolved_lookup.get(row["post_id"])
    if resolved is None:
        continue
    if resolved["platform"] not in ("bluesky", "reddit"):
        continue
    ro_id = post_to_ro.get(row["post_id"])
    if ro_id not in papers_df.index:
        continue
    p = papers_df.loc[ro_id]
    post = ResolvedPost(
        post_id=resolved["post_id"], platform=resolved["platform"], text=resolved["text"],
        author_handle=resolved.get("author_handle"), author_id=resolved.get("author_id"),
        url=resolved["url"], created_at=resolved["created_at"],
        raw_title=resolved["raw_title"], text_confidence=resolved["text_confidence"],
    )
    paper = PaperCtx(title=str(p["title"]), abstract=p.get("abstract"))
    endo = Endorsement(
        claim_text=row["claim_text"] or "",
        magnitude_dB=int(row["magnitude_dB"]),
        criterion=row["criterion"] or "Support",
        reasoning=row["reasoning"] or "",
    )
    seeds.append(TraversalSeed(post=post, paper=paper, result=endo))

print(f"Built {len(seeds)} traversal seeds (Bluesky/Reddit endorsements only)")

## Run the traversal

Depth-1 fan-out over the seeds. **Jupyter has its own running event loop, so we use top-level `await` — DO NOT use `asyncio.run`.**

In [ ]:
import asyncio

import aiohttp
import anthropic
import nest_asyncio

# Patch Jupyter's running event loop so asyncio.run() works inside cells.
nest_asyncio.apply()

from altendor.enrich.reddit import get_reddit_client
from altendor.traverse.replies import traverse_depth1

client = anthropic.Anthropic()
reddit_client = get_reddit_client()


async def go():
    async with aiohttp.ClientSession() as bsky:
        return await traverse_depth1(client, seeds, bsky_session=bsky, reddit_client=reddit_client)


traversal_rows = asyncio.run(go())
print(f"Discovered + classified {len(traversal_rows)} reply rows")

## Append rows back

Each traversed reply becomes one new row in `classified.parquet` (re-classified through the same schema) and one new row in `resolved_posts.parquet` (so downstream notebooks can find it).

In [ ]:
from dataclasses import asdict
from altendor.classify.schema import Endorsement, Flag, Irrelevant

new_classified_rows = []
new_resolved_rows = []
for tr in traversal_rows:
    reply = tr.reply
    new_resolved_rows.append(asdict(reply))
    ro_id = post_to_ro.get(tr.parent_post_id) or papers_df.index[0]  # heuristic; reply inherits parent's paper
    doi = papers_df.loc[ro_id, "doi"] if ro_id in papers_df.index else None
    base = {"post_id": reply.post_id, "ro_id": ro_id, "doi": doi,
            "kind": None, "claim_text": None, "magnitude_dB": None,
            "criterion": None, "reasoning": None,
            "category": None, "rationale": None, "reason": None,
            "error_reason": None}
    if isinstance(tr.result, Endorsement):
        base.update(kind="endorsement", claim_text=tr.result.claim_text,
                    magnitude_dB=tr.result.magnitude_dB, criterion=tr.result.criterion,
                    reasoning=tr.result.reasoning)
    elif isinstance(tr.result, Flag):
        base.update(kind="flag", category=tr.result.category, rationale=tr.result.rationale)
    elif isinstance(tr.result, Irrelevant):
        base.update(kind="irrelevant", reason=tr.result.reason)
    new_classified_rows.append(base)

if new_classified_rows:
    classified_df = pd.concat([classified_df, pd.DataFrame(new_classified_rows)], ignore_index=True)
    classified_df.to_parquet(OUT_DIR / "classified.parquet", index=False)
if new_resolved_rows:
    resolved_df = pd.concat([resolved_df, pd.DataFrame(new_resolved_rows)], ignore_index=True)
    resolved_df.to_parquet(OUT_DIR / "resolved_posts.parquet", index=False)
print(f"Appended {len(new_classified_rows)} classified rows + {len(new_resolved_rows)} resolved rows.")

In [ ]:
import json
manifest_path = OUT_DIR / "manifest.json"
manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
manifest["stage_6_traverse_replies"] = {
    "n_seeds": len(seeds),
    "n_replies_classified": len(traversal_rows),
}
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True))
print("Manifest updated.")